#  Movie Recommender System

Clean, structured notebook for building a content-based movie recommender using `movies_metadata.csv`.

This project recommends movies similar to a given movie using metadata-based text features and cosine similarity.

## 1. Project Overview

This project is a **Content-Based Movie Recommender System**. It recommends similar movies based on movie metadata such as:

- Title
- Overview
- Genres
- Tagline

The system converts movie information into text features, transforms those features using **TF-IDF Vectorization**, and calculates similarity using **Cosine Similarity**.

## 2. Dataset Information

The dataset used in this project is:

`movies_metadata.csv`

Important columns used:

- `title` — Movie name
- `overview` — Movie description/summary
- `genres` — Movie genres stored in JSON-like format
- `tagline` — Short promotional movie tagline

Only the required columns are selected to keep the recommender clean and efficient.

## 3. Importing Libraries

All required libraries are imported for data handling, text preprocessing, vectorization, similarity calculation, and saving files.

In [46]:
import pandas as pd
import numpy as np
import ast
import re
import difflib
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

## 4. Loading the Data

The dataset is loaded using Pandas. `low_memory=False` is used because this dataset may contain mixed data types.

In [47]:
df = pd.read_csv("movies_metadata.csv", low_memory=False)
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


## 5. Data Understanding

Basic checks are performed to understand the dataset size, columns, data types, and missing values.

In [48]:
print("Shape:", df.shape)
df.info()

Shape: (45466, 24)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-nu

In [49]:
df.isnull().sum().sort_values(ascending=False).head(20)

belongs_to_collection    40972
homepage                 37684
tagline                  25054
overview                   954
poster_path                386
runtime                    263
status                      87
release_date                87
imdb_id                     17
original_language           11
spoken_languages             6
title                        6
video                        6
vote_average                 6
revenue                      6
vote_count                   6
popularity                   5
production_companies         3
production_countries         3
original_title               0
dtype: int64

## 6. Data Cleaning

Only the columns required for the recommendation system are selected.


In [50]:
df = df[['title', 'overview', 'genres', 'tagline']].copy()

df['overview'] = df['overview'].fillna("")
df['tagline'] = df['tagline'].fillna("")

df = df.dropna(subset=['title', 'genres'])

df['clean_title'] = df['title'].str.lower().str.strip()
df = df.drop_duplicates(subset='clean_title').reset_index(drop=True)

df.head()

,title,overview,genres,tagline,clean_title
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",,toy story
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Roll the dice and unleash the excitement!,jumanji
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",Still Yelling. Still Fighting. Still Ready for...,grumpier old men
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",Friends are the people who let you be yourself...,waiting to exhale
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]",Just When His World Is Back To Normal... He's ...,father of the bride part ii


## 7. Feature Engineering


In [51]:
def convert_genres(obj):
    genres = []
    try:
        for item in ast.literal_eval(obj):
            genres.append(item['name'])
    except:
        return ""
    return " ".join(genres)

df['genres'] = df['genres'].apply(convert_genres)
df[['title', 'genres']].head()

,title,genres
0,Toy Story,Animation Comedy Family
1,Jumanji,Adventure Fantasy Family
2,Grumpier Old Men,Romance Comedy
3,Waiting to Exhale,Comedy Drama Romance
4,Father of the Bride Part II,Comedy


## 8. Creating Tags

A new column named `tags` is created by combining the main textual features.

This combined text represents each movie and is used for similarity comparison.

In [52]:
df['tags'] = (
    df['title'] + " " +df['overview'] + " " + df['genres'] + " " + df['tagline']
)

df[['title', 'tags']].head()

,title,tags
0,Toy Story,"Toy Story Led by Woody, Andy's toys live happi..."
1,Jumanji,Jumanji When siblings Judy and Peter discover ...
2,Grumpier Old Men,Grumpier Old Men A family wedding reignites th...
3,Waiting to Exhale,"Waiting to Exhale Cheated on, mistreated and s..."
4,Father of the Bride Part II,Father of the Bride Part II Just when George B...


## 9. Text Preprocessing

The text is cleaned by:

- Converting to lowercase
- Removing punctuation/special characters
- Removing extra spaces

In [53]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['tags'] = df['tags'].apply(preprocess_text)
df[['title', 'tags']].head()

,title,tags
0,Toy Story,toy story led by woody andy s toys live happil...
1,Jumanji,jumanji when siblings judy and peter discover ...
2,Grumpier Old Men,grumpier old men a family wedding reignites th...
3,Waiting to Exhale,waiting to exhale cheated on mistreated and st...
4,Father of the Bride Part II,father of the bride part ii just when george b...


## 10. Vectorization

The cleaned text is converted into numerical vectors using **TF-IDF Vectorizer**.

TF-IDF gives more importance to meaningful words and reduces the importance of very common words.

In [54]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(df['tags'])

tfidf_matrix.shape

(42227, 50000)

## 11. Title Index Mapping

A clean title index is created so the recommendation function can correctly find movies even if the user types uppercase/lowercase differently.

In [55]:
indices = pd.Series(df.index, index=df['clean_title']).drop_duplicates()
'jumanji' in indices

True

## 12. Recommendation Function

The recommendation function:

1. Cleans the input title
2. Checks whether the movie exists
3. Suggests close matches if the exact movie is not found
4. Calculates cosine similarity between the selected movie and all other movies
5. Returns the top similar movies

In [56]:
def recommend(title, n=10):
    title = title.lower().strip()

    if title not in indices:
        matches = difflib.get_close_matches(title, indices.index, n=5, cutoff=0.4)
        if matches:
            suggestions = [df.loc[indices[m], 'title'] for m in matches]
            return [f"Movie not found. Did you mean: {suggestions}?"]
        return ["Movie not found"]

    idx = indices[title]

    sim_scores = cosine_similarity(tfidf_matrix[idx:idx+1], tfidf_matrix).flatten()
    similar_indices = sim_scores.argsort()[::-1][1:n+1]

    recommendations = []

    for i in similar_indices:
        recommendations.append({
            "title": df.iloc[i]['title'],
            "similarity_score": round(float(sim_scores[i]), 3)
        })

    return recommendations

## 13. Testing the Recommender

Test the recommender system with sample movie names.

In [57]:
recommend("Jumanji")

[{'title': 'Aschenputtel', 'similarity_score': 0.168},
 {'title': 'Table No. 21', 'similarity_score': 0.166},
 {'title': 'Word Wars', 'similarity_score': 0.159},
 {'title': 'Game Over', 'similarity_score': 0.155},
 {'title': 'Liar Game: Reborn', 'similarity_score': 0.151},
 {'title': 'The Ouija Exorcism', 'similarity_score': 0.147},
 {'title': 'Snowed Under', 'similarity_score': 0.145},
 {'title': 'Le Pont du Nord', 'similarity_score': 0.14},
 {'title': 'A Serious Game', 'similarity_score': 0.137},
 {'title': 'Quintet', 'similarity_score': 0.136}]

In [58]:
recommend("Avatar")

[{'title': 'Avatar 2', 'similarity_score': 0.348},
 {'title': 'Avatar: Creating the World of Pandora', 'similarity_score': 0.285},
 {'title': 'The Inhabited Island', 'similarity_score': 0.182},
 {'title': 'Moontrap: Target Earth', 'similarity_score': 0.158},
 {'title': 'Pandora and the Flying Dutchman', 'similarity_score': 0.154},
 {'title': 'The Matrix', 'similarity_score': 0.153},
 {'title': 'The Secret of the Third Planet', 'similarity_score': 0.152},
 {'title': 'Stand by Me Doraemon', 'similarity_score': 0.15},
 {'title': 'A Trip to the Moon', 'similarity_score': 0.148},
 {'title': 'On the Silver Globe', 'similarity_score': 0.141}]

## 14. Saving Files for Deployment

The cleaned movie dataframe and TF-IDF matrix are saved using pickle.

These files can be used later in the Streamlit app.

In [59]:
pickle.dump(df, open("movie_dict.pkl", "wb"))
pickle.dump(tfidf_matrix, open("tfidf_matrix.pkl", "wb"))
pickle.dump(tfidf, open("tfidf_vectorizer.pkl", "wb"))

## 15. Conclusion

This project demonstrates how NLP and similarity-based techniques can be used to build a content-based movie recommender system.

Key learnings:

- Data cleaning and preprocessing
- Feature engineering from movie metadata
- TF-IDF vectorization
- Cosine similarity
- Recommendation function development
- Saving model files for deployment
